In [ ]:
# Week 2 Day 6: Support Vector Machine

### Today's Goal

Today I will learn the basic idea of Support Vector Machine and compare it with Logistic Regression and Random Forest

## Learning Objectives

- Understand the idea of margin
- Understand support vectors
- Train an SVM classifier using sklearn
- Explain why SVM needs feature scaling
- Compare SVM with previous models

In [2]:
# Step 1: Load the Breast Cancer dataset

from sklearn.datasets import load_breast_cancer
import numpy as np
import pandas as pd

data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target names:", data.target_names)
print("Feature count:", len(data.feature_names))
print("Class counts:", np.bincount(y))

X shape: (569, 30)
y shape: (569,)
Target names: ['malignant' 'benign']
Feature count: 30
Class counts: [212 357]


In [5]:
# Step 2: Train/Test Split and Feature Scaling

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Train class counts:", np.bincount(y_train))
print("Test class counts:", np.bincount(y_test))

print("First three scaled train means:", X_train_scaled.mean(axis=0)[:3])
print("First three scaled train stds:", X_train_scaled.std(axis=0)[:3])

X_train shape: (455, 30)
X_test shape: (114, 30)
Train class counts: [170 285]
Test class counts: [42 72]
First three scaled train means: [-4.31742554e-15  2.24606658e-15 -7.38359313e-16]
First three scaled train stds: [1. 1. 1.]


In [ ]:
## Prediction Before Running

1. Will SVM train accuracy be high or low?
high
2. Will SVM test accuracy be close to Logistic Regression?
it will
3. Does `SVC(kernel="linear")` learn a linear or non-linear decision boundary?
linear
4. In this dataset, are Precision and Recall currently calculated for malignant or benign?
benign, because benign = 1 in this dataset

In [7]:
# Step 3: Train a linear SVM model

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

svm_model = SVC(
    kernel="linear",
    C=1.0,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

y_train_pred_svm = svm_model.predict(X_train_scaled)
y_test_pred_svm = svm_model.predict(X_test_scaled)

print("SVM train accuracy:", accuracy_score(y_train, y_train_pred_svm))
print("SVM test accuracy:", accuracy_score(y_test, y_test_pred_svm))
print("SVM test precision:", precision_score(y_test, y_test_pred_svm))
print("SVM test recall:", recall_score(y_test, y_test_pred_svm))
print("SVM test F1:", f1_score(y_test, y_test_pred_svm))

SVM train accuracy: 0.9912087912087912
SVM test accuracy: 0.9736842105263158
SVM test precision: 0.9859154929577465
SVM test recall: 0.9722222222222222
SVM test F1: 0.9790209790209791


In [8]:
# Step 4: Compare different C values

c_values = [0.01, 0.1, 1, 10, 100]

svm_results = []

for c in c_values:
    model = SVC(
        kernel="linear",
        C=c,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)

    svm_results.append({
        "C": c,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred),
        "Recall": recall_score(y_test, test_pred),
        "F1": f1_score(y_test, test_pred)
    })

svm_results_df = pd.DataFrame(svm_results)
svm_results_df

,C,Train Accuracy,Test Accuracy,Precision,Recall,F1
0,0.01,0.978022,0.964912,0.959459,0.986111,0.972603
1,0.10,0.984615,0.982456,0.986111,0.986111,0.986111
2,1.00,0.991209,0.973684,0.985915,0.972222,0.979021
3,10.00,0.991209,0.982456,0.986111,0.986111,0.986111
4,100.00,0.995604,0.973684,0.972603,0.986111,0.979310


In [11]:
# Step 5: Compare Logistic Regression, Random Forest and SVM

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=10000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="linear", C=0.1, random_state=42))
    ])
}

comparison_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    comparison_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.982456,0.986111,0.986111,0.986111
1,Random Forest,0.956140,0.958904,0.972222,0.965517
2,SVM,0.982456,0.986111,0.986111,0.986111


In [ ]:
# Reflection

## 1. What did I learn today?
I learnt the model called SVM which finds a decision boundary with a large margin and I compared three models with metrics
## 2. What does the parameter C control in SVM?
Parameter C control the torlerance 
Smaller C -- more torlerance -- wider margin -- stronger regularization
Larger C -- less torlerance -- narrower margin -- more likely overfitting
## 3. Why does this result not prove that SVM or Logistic Regression is always the best model?
because this reults just proved that SVM or Logistic Regression performs better in the dataset and we can not extend to other datasets